# Código 1: Detección y clasificación de outliers (sensor_error vs evento_real)

In [ ]:
# Código 1: Detección y clasificación de outliers (sensor_error vs evento_real)
# Diseñado para ejecutarse en un notebook (VSCode, Python 3.11).
# Dependencias: pandas, numpy, scikit-learn, statsmodels
# Instalar si es necesario: pip install pandas numpy scikit-learn statsmodels

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ---------------------------
# Parámetros configurables
# ---------------------------
PERIOD_H = 24                      # periodo estacional (horario → 24h)
STL_K_MAD = 4.0                    # umbral en MAD para residuo STL
JUMP_K_MAD = 6.0                   # umbral para saltos 1h basados en MAD local
ISO_CONTAMINATION = 0.01           # proporción para IsolationForest
ROLL_WINDOW_H = 24                 # ventana (horas) para calcular MAD local
PHYS_TEMP_MIN = -50.0
PHYS_TEMP_MAX = 60.0
PHYS_RSOL_MIN = 0.0
PHYS_WIND_MIN = 0.0
ANGLE_MIN = 0.0
ANGLE_MAX = 360.0
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")  # carpeta salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Lista de rutas (usa la tuya)
# ---------------------------
station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
station_paths = [os.path.expanduser(p) for p in station_paths]

# ---------------------------
# Funciones auxiliares
# ---------------------------
def robust_mad(x):
    """MAD: median absolute deviation (no-Bessel)."""
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return mad if mad > 0 else np.nanmean(np.abs(x - med))  # fallback

def ensure_datetime_index(df, datetime_col=None, freq='h'):
    """Asegura índice datetime con frecuencia horaria y reindexa dejando NaNs donde faltan timestamps."""
    if datetime_col is None:
        # intentar encontrar columna timestamp
        candidates = [c for c in df.columns if c.lower() in ("datetime","fecha","time","timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Pasa el nombre con datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    # rellenar para rango completo
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df


def physical_checks(df):
    """Marca errores físicos obvios."""
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['reason'] = ""
    # R.Sol. < 0
    if 'R.Sol.' in df.columns:
        mask = (df['R.Sol.'] < PHYS_RSOL_MIN) & (~df['R.Sol.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "R.Sol. < 0; "
    # Veloc < 0
    if 'Veloc.' in df.columns:
        mask = (df['Veloc.'] < PHYS_WIND_MIN) & (~df['Veloc.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Veloc < 0; "
    # Angulo/Direc fuera de rango
    for col in ['Direc.','Angulo(en grados)','Angle','Dirección']:
        if col in df.columns:
            mask = (~df[col].isna()) & ((df[col] < ANGLE_MIN) | (df[col] >= ANGLE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{col} fuera [0,360); "
    # Temp extrema
    if 'Temp' in df.columns:
        mask = (~df['Temp'].isna()) & ((df['Temp'] < PHYS_TEMP_MIN) | (df['Temp'] > PHYS_TEMP_MAX))
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Temp extrema; "
    # O3 negativa
    for o3_col in ['O3','O₃','Ozone']:
        if o3_col in df.columns:
            mask = (~df[o3_col].isna()) & (df[o3_col] < 0)
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{o3_col} < 0; "
    return flags

def stl_outliers(df, col='O3', period=PERIOD_H, k=STL_K_MAD):
    """Detecta outliers en la serie usando STL residuo + MAD"""
    res = pd.Series(False, index=df.index)
    try:
        s = df[col].dropna()
        if len(s) > period*2:
            stl = STL(s, period=period, robust=True)
            out = stl.fit()
            resid = out.resid
            mad_resid = robust_mad(resid)
            threshold = k * mad_resid
            mask = np.abs(resid) > threshold
            res.loc[mask.index] = mask.values
    except Exception as e:
        # si falla, devolvemos serie False
        print(f"STL failed for {col}: {e}")
    return res

def rolling_jump_outliers(df, col='O3', window=ROLL_WINDOW_H, k=JUMP_K_MAD):
    """Detecta saltos bruscos 1h comparando diffs y MAD local."""
    res = pd.Series(False, index=df.index)
    s = df[col]
    # diff 1h (forward/backward)
    diff = s.diff().abs()
    # calc rolling MAD of diffs
    roll_mad = diff.rolling(window=window, min_periods=1, center=True).apply(lambda x: robust_mad(x), raw=True)
    thr = k * roll_mad
    mask = (diff > thr) & (~diff.isna())
    res.loc[mask.index] = mask.values
    return res

def isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION):
    """IsolationForest multivariado (retorna boolean series)."""
    if features is None:
        # elegir columnas relevantes si existen
        candidates = ['O3','NO','NO2','NOx','Temp','Veloc.','R.Sol.']
        features = [c for c in candidates if c in df.columns]
    # require enough rows
    res = pd.Series(False, index=df.index)
    sub = df[features].copy()
    # need to drop rows with all-NaN for features
    valid_idx = sub.dropna(how='all').index
    if len(valid_idx) < 20:
        return res
    sub = sub.loc[valid_idx].fillna(method='ffill').fillna(method='bfill').fillna(0)
    iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
    try:
        pred = iso.fit_predict(sub.values)
        # -1 are outliers
        out_idx = sub.index[pred == -1]
        res.loc[out_idx] = True
    except Exception as e:
        print("IsolationForest failed:", e)
    return res

def neighbor_support(df, idx, col='O3', window=3, rel_tol=0.5):
    """
    Evalúa si un pico en idx tiene soporte en vecinos (vecinos muestran cambios coherentes).
    window: número total de puntos a mirar (debe ser impar), default 3 -> t-1, t, t+1
    rel_tol: fracción del tamaño del cambio que consideraremos 'soporte'
    """
    half = (window - 1) // 2
    i_pos = df.index.get_loc(idx)
    start = max(0, i_pos - half)
    end = min(len(df) - 1, i_pos + half)
    vals = df[col].iloc[start:end+1].values
    center = df[col].iloc[i_pos]
    # si la mayoría de vecinos están NaN → no hay soporte
    if np.isnan(vals).sum() >= half + 1:
        return False
    # support heuristic: si cambio del centro respecto a median vecino es mayor que
    # median(neighbors) * rel_tol -> we consider neighbors support if they change similarly
    med_neighbors = np.nanmedian(np.concatenate([vals[:half], vals[half+1:]])) if len(vals) > 1 else np.nan
    if np.isnan(med_neighbors):
        return False
    # if both neighbors changed in same direction as center relative to older values -> support
    # Simpler: if abs(center - med_neighbors) < abs(center) * rel_tol => we say has support
    if abs(center - med_neighbors) <= abs(center) * rel_tol:
        return True
    return False

# ---------------------------
# Pipeline para una estación
# ---------------------------
def process_station(path, save_out=True, verbose=True):
    """Carga CSV de estación, detecta outliers y clasifica sensor_error/evento_real/doubtful.
       Devuelve dataframe con columnas nuevas y guarda CSV con sufijo _outliers.csv"""
    station_name = Path(path).stem
    if verbose:
        print("Procesando:", station_name)
    df_raw = pd.read_csv(path)
    df = ensure_datetime_index(df_raw, datetime_col=None, freq='H')
    # Inicializar flags
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""
    # physical checks
    phys = physical_checks(df)
    flags['physical_error'] = phys['physical_error']
    flags['physical_reason'] = phys['reason']

    # detectar outliers univariados por STL (O3 si existe)
    o3_col = next((c for c in df.columns if c in ['O3','O₃','Ozone']), None)
    if o3_col is None:
        print(f"Warning: {station_name} no tiene columna O3 reconocida. Saltando STL/jump para O3.")
        stl_mask = pd.Series(False, index=df.index)
        jump_mask = pd.Series(False, index=df.index)
    else:
        stl_mask = stl_outliers(df, col=o3_col, period=PERIOD_H, k=STL_K_MAD)
        jump_mask = rolling_jump_outliers(df, col=o3_col, window=ROLL_WINDOW_H, k=JUMP_K_MAD)

    flags['outlier_stl'] = stl_mask
    flags['outlier_jump'] = jump_mask

    # Isolation Forest multivariado
    iso_mask = isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION)
    flags['outlier_iso'] = iso_mask

    # Clasificación heurística sensor_error vs evento_real vs doubtful
    flags['sensor_error'] = False
    flags['evento_real'] = False
    flags['doubtful'] = False

    # reglas básicas
    for ts in df.index:
        reasons = []
        # physical error overrides
        if flags.at[ts, 'physical_error']:
            flags.at[ts, 'sensor_error'] = True
            reasons.append("physical")
            continue
        # detector combinados
        is_stl = bool(flags.at[ts, 'outlier_stl'])
        is_jump = bool(flags.at[ts, 'outlier_jump'])
        is_iso = bool(flags.at[ts, 'outlier_iso'])
        any_det = is_stl or is_jump or is_iso
        if not any_det:
            # no outlier -> nothing to do
            continue
        # if multiple detectors agree → more fiable
        n_det = sum([is_stl, is_jump, is_iso])
        # check neighbour support only for O3
        support = False
        if o3_col is not None and not np.isnan(df.at[ts,o3_col]):
            try:
                support = neighbor_support(df, ts, col=o3_col, window=3, rel_tol=0.6)
            except Exception:
                support = False
        # Heurística:
        # - Si hay detección pero sin soporte en vecinos y detección es 'aislada' -> probable sensor_error
        # - Si varios detectores y hay soporte / hay picos también en NO/NO2/NOx/Temp/R.Sol -> evento_real
        related_vars = ['NO','NO2','NOx','Temp','R.Sol.','Veloc.']
        related_present = any([(v in df.columns) and (not pd.isna(df.at[ts,v])) for v in related_vars])
        # check if related vars also spike (simple: compare to rolling median)
        related_spike = False
        for v in related_vars:
            if v in df.columns:
                val = df.at[ts,v]
                if pd.isna(val):
                    continue
                rolling_med = df[v].rolling(window=ROLL_WINDOW_H, min_periods=1, center=True).median().iloc[df.index.get_loc(ts)]
                if not pd.isna(rolling_med):
                    if abs(val - rolling_med) > 2 * robust_mad(df[v].dropna()):  # simple threshold
                        related_spike = True
                        break
        # classification rules:
        if n_det >= 2 and related_spike:
            flags.at[ts, 'evento_real'] = True
            reasons.append("multi_det + related_spike")
        else:
            # if isolated detection and no neighbor support => sensor error
            if not support and n_det >= 1:
                flags.at[ts, 'sensor_error'] = True
                reasons.append("isolated_no_support")
            else:
                # ambiguous: iso true but no clear support or multiple detectors but no related spike
                flags.at[ts, 'doubtful'] = True
                reasons.append("ambiguous")
        # end loop

    # Crear columna de imputation: si sensor_error True -> poner NaN (preparar imputación)
    df_proc = df.copy()
    if o3_col is not None:
        df_proc['O3_for_impute'] = df_proc[o3_col].where(~flags['sensor_error'], np.nan)

    # adjuntar flags al df final
    out_df = pd.concat([df_proc, flags], axis=1)

    # guardar CSV con sufijo
    if save_out:
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_outliers.csv")
        out_df.to_csv(out_path, index=True)
        if verbose:
            print(f"Guardado: {out_path}")
    return out_df

# ---------------------------
# Procesar todas las estaciones y compilar doubtfuls
# ---------------------------
all_doubtful = []
all_summaries = []
for p in station_paths:
    try:
        df_out = process_station(p, save_out=True, verbose=True)
        # extraer filas dudosas
        if 'doubtful' in df_out.columns:
            doubtful_rows = df_out[df_out['doubtful'] == True].copy()
            if not doubtful_rows.empty:
                # añadir columna estación
                doubtful_rows['station_file'] = Path(p).stem
                all_doubtful.append(doubtful_rows)
        # summary counts
        summary = {
            'station': Path(p).stem,
            'n_rows': len(df_out),
            'n_physical_error': int(df_out['physical_error'].sum()) if 'physical_error' in df_out.columns else 0,
            'n_outlier_stl': int(df_out['outlier_stl'].sum()) if 'outlier_stl' in df_out.columns else 0,
            'n_outlier_jump': int(df_out['outlier_jump'].sum()) if 'outlier_jump' in df_out.columns else 0,
            'n_outlier_iso': int(df_out['outlier_iso'].sum()) if 'outlier_iso' in df_out.columns else 0,
            'n_sensor_error': int(df_out['sensor_error'].sum()) if 'sensor_error' in df_out.columns else 0,
            'n_evento_real': int(df_out['evento_real'].sum()) if 'evento_real' in df_out.columns else 0,
            'n_doubtful': int(df_out['doubtful'].sum()) if 'doubtful' in df_out.columns else 0,
        }
        all_summaries.append(summary)
    except Exception as e:
        print(f"Error procesando {p}: {e}")

# guardar doubtfuls combinados
if all_doubtful:
    combined = pd.concat(all_doubtful, axis=0)
    combined_path = os.path.join(OUTPUT_DIR, "doubtful_cases_combined.csv")
    combined.to_csv(combined_path, index=True)
    print(f"CSV combinado de casos dudosos guardado en: {combined_path}")
# guardar resumen
summary_df = pd.DataFrame(all_summaries)
summary_path = os.path.join(OUTPUT_DIR, "outliers_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

print("Proceso completado. Revisa la carpeta de salida para CSVs por estación, summary y casos dudosos.")

Procesando: T1_E1_Alicante
